# Putting objects on the board while the sim runs

MuJoCo freezes `nbody`/`njnt`/`ngeom` at compile time — there is no `add_body()`,
which is why `server.py` still refuses a runtime `scene_load`.  So "add an
object" means "move one that already exists".  `FWDCenterLabSivaPool.yaml`
declares ten of them, parked on the floor beside the table and out of the arm's
reach, and this notebook moves them onto named grid cells.

Start the sim with that scene first:

```
cd native_mujoco && mjpython server.py --scene ../scenes/FWDCenterLabSivaPool.yaml
```

In [ ]:
import asyncio, json, sys, pathlib
import websockets

sys.path.insert(0, str(pathlib.Path.cwd().parent / "native_mujoco"))
from protocol import Hello, PlaceObject

URI = "ws://127.0.0.1:8765"

## A tiny client

One connection, and a helper that waits for the ack rather than firing and
hoping.  The ack carries `sim_step` — the step the object actually landed on —
which is the whole point: a client that knows the step can line the next camera
frame up with a placement whose position it already knows.  That is what makes
this a perception check and not a demo.

In [ ]:
class Board:
    def __init__(self, ws):
        self.ws = ws

    async def _await_ack(self, request_id):
        while True:
            msg = json.loads(await self.ws.recv())
            if msg["type"] == "place_ack" and msg["request_id"] == request_id:
                return msg

    async def place(self, object_id, cell, yaw_deg=0.0, reshape=None, **kw):
        rid = f"{object_id}-{cell}"
        await self.ws.send(PlaceObject(object_id=object_id, cell=cell,
                                       yaw_deg=yaw_deg, reshape=reshape,
                                       request_id=rid, **kw).encode())
        ack = await self._await_ack(rid)
        if not ack["accepted"]:
            raise RuntimeError(ack["error"])
        return ack

    async def stow(self, object_id):
        rid = f"{object_id}-stow"
        await self.ws.send(PlaceObject(object_id=object_id, cell=None,
                                       request_id=rid).encode())
        return await self._await_ack(rid)

    async def poses(self):
        while True:
            msg = json.loads(await self.ws.recv())
            if msg["type"] == "state":
                return {o["object_id"]: o["pos_xyz"] for o in msg["objects"]}

## Set a board up

Nothing starts on the grid — this scene parks its manipulables with the rest of
the pool, so whatever is on the table has a caller who put it there.

In [ ]:
async def setup():
    async with websockets.connect(URI, max_size=None) as ws:
        await ws.send(Hello().encode()); await ws.recv()
        board = Board(ws)

        for oid, cell in [("red_cube", "r1c3"),
                          ("soda_can", "r2c2"),
                          ("foam_block", "r1c1")]:
            ack = await board.place(oid, cell)
            p = ack["placement"]
            print(f"{oid:12s} -> {p['cell']}  z={p['position'][2]:.3f}  "
                  f"step={ack['sim_step']}")

await setup()

## What it refuses, and why that matters

`cell_r3c1` and `cell_r3c2` were measured unreachable for the right arm by
40-restart IK in `FWDCenterLabMCC` and confirmed physically.  Placing a target
there gives the planner something it can never pick, so it is refused by
default — with the distance in the message, not a bare exception.

In [ ]:
async def refusals():
    async with websockets.connect(URI, max_size=None) as ws:
        await ws.send(Hello().encode()); await ws.recv()
        board = Board(ws)

        for oid, cell, kw in [("pool_box_1", "r3c1", {}),
                              ("pool_box_1", "r1c3", {})]:     # r1c3 is taken
            try:
                await board.place(oid, cell, **kw)
                print(f"{cell}: placed")
            except RuntimeError as exc:
                print(f"{cell}: refused — {exc}")

        # both refusals are overridable when the refusal is the point
        await board.place("pool_box_1", "r3c1", allow_unreachable=True)
        print("r3c1: placed deliberately, to test that a planner declines")

await refusals()

## Reshaping a slot

`geom_size`, `geom_rgba` and `body_mass` are writable on a compiled model, so a
slot becomes whatever the run needs without a recompile.

Always go through `reshape` rather than writing `geom_size` yourself.
`geom_rbound` is the broadphase bounding radius, it is computed by the compiler,
and it is **not** recomputed when `geom_size` changes — `mj_setConst` does not
fix it either.  Grow a geom without updating rbound and the broadphase culls the
pair before narrowphase runs, so contacts silently stop being generated.  That
shows up as the gripper passing through the object, with nothing in any log.

In [ ]:
async def reshape():
    async with websockets.connect(URI, max_size=None) as ws:
        await ws.send(Hello().encode()); await ws.recv()
        board = Board(ws)

        # a 4 cm cube on the centre cell, in the achromatic grey the real
        # objects photograph as
        await board.place("pool_box_2", "r2c1",
                          reshape={"size": [0.04, 0.04, 0.04],
                                   "rgba": [0.46, 0.46, 0.45, 1.0],
                                   "mass": 0.05})
        print("placed a 4 cm cube on r2c1")

        poses = await board.poses()
        for oid in sorted(poses):
            x, y, z = poses[oid]
            where = "board" if z > 0.5 else "floor"
            print(f"  {oid:16s} ({x:+.3f}, {y:+.3f}, {z:.3f})  {where}")

await reshape()

## Clearing up

In [ ]:
async def clear():
    async with websockets.connect(URI, max_size=None) as ws:
        await ws.send(Hello().encode()); await ws.recv()
        board = Board(ws)
        for oid in ("red_cube", "soda_can", "foam_block",
                    "pool_box_1", "pool_box_2"):
            await board.stow(oid)
        print("board clear")

await clear()